----
## <font color='CornflowerBlue'>Practical 5: Introduction to transformers</font> 
##### Alok Bharadwaj and Arjen Jakobi
---


In the previous notebook we saw how raw text and protein sequences can be converted into sequences of numerical tokens, and how those tokens are represented as embedding vectors. We ended with a key question: how can a model use the *context* around a token to refine its meaning?

This notebook answers that question by building the **transformer** architecture from scratch. We will progress through four levels of abstraction:

1. **Self-attention**: the mechanism by which tokens exchange information with one another, computed as scaled dot products between learned query and key projections.
2. **Multi-head attention**: running several independent attention heads in parallel so the model can attend to different types of relationships simultaneously.
3. **Transformer block**: wrapping attention and a feed-forward network inside residual connections and layer normalisation.
4. **Language model**: stacking transformer blocks into a full model with token and positional embeddings, and applying it to a real protein sequence.

Each level builds directly on the one before. Interactive visualisations let you manipulate the weight matrices and observe how attention patterns change, before the same operations are assembled into working PyTorch code.

> **Interactive reference:** [Transformer Explainer](https://poloclub.github.io/transformer-explainer/) — a visual walkthrough of the full transformer architecture. Refer to it alongside this notebook to build a geometric intuition for each component.

### 5.4: Self-attention

Now that we understand how sequence information can be represented as vector embeddings, we need a mechanism that allows tokens to exchange information with one another based on context. That mechanism is **self-attention**.

Consider a sequence of $N$ tokens, each represented by an embedding vector of size $d$:

\begin{equation*}
    x = \{ x_1, x_2, x_3, \ldots, x_N \}
\end{equation*}

This set is called the **context window**. Each element $x_i \in \mathbb{R}^{d}$ is a row vector, where $N$ is the maximum number of tokens that can be accommodated in the context.

An important property of sequence data is **long-range dependency**: the meaning of a token $x_i$ can depend on *any* other token in the context, regardless of distance. A model that only looks at adjacent tokens (like a convolutional network) cannot capture this. Self-attention solves the problem by allowing every token to directly attend to every other token.

#### 5.4.1 Computing attention as a dot product
Long range dependancy between two input embeddings are computed as attention scores between them. Attention between two embedding vectors are computed as the dot product between them. When two vectors point in the same direction, they have higher dot product and therefore higher attention scores. 

You probably remember the formula for dot product:

\begin{equation*}
    \vec{u} \cdot \vec{v} = |u| |v| cos(\theta)
\end{equation*}

where $\theta$ is the angle between two vectors. Play with the interactive sliders below to get an intuition for the relationship between alignment and the dot product between two vectors. 

In [ ]:
from src.utils import interactive_dot_product

interactive_dot_product()

You should realise that:

- Vectors pointing in the **same direction** have a high dot product → high attention
- vectors that are **orthogonal** to each other have a dot product of zero → no attention
- vectors pointing in the **opposite direction** have a high negative dot product → negative attention score

The dot product therefore acts as a geometric similarity measure. During training, tokens with related meanings or grammatical roles learn to have similar embedding directions, so they naturally attend to each other.


#### 5.4.2 Projecting inputs to query, key, and value space

Self-attention works by projecting each input embedding into three separate vector spaces:

- <font color='CornflowerBlue'>**Query**</font> ($q_i$): *What information does this token need?*
- <font color='Red'>**Key**</font> ($k_i$): *What information does this token provide?*
- <font color='Green'>**Value**)</font> ($v_i$): *What information does this token share when selected?*

The attention score between two tokens is the dot product of one token's query with another token's key: how well does what token $i$ needs match what token $j$ offers? Value vectors are then blended using these scores as weights.

All three projections use learned weight matrices $W_q$, $W_k$, and $W_v$. Query and key vectors are projected to a lower-dimensional *head dimension* $d_h \leq d$, making attention cheaper to compute. The weight matrices for $W_q$ and $W_k$ therefore have shape $(d, d_h)$. The value matrix $W_v$ has shape $(d, d_v)$, and typically $d_v = d_h$.

Let us first look at query $q_i$ and key $k_i$. For an input token $x_i$:

\begin{equation*}
    q_i = x_i W_q, \qquad k_i = x_i W_k
\end{equation*}


In [ ]:
import numpy as np
from src.utils import interactive_qk_vectors
x_1 = np.array([1.0, 0.5, -0.8])
x_2 = np.array([-0.4, 1.1, 1.2])
W_q = np.array([[0.92, -0.89], [0.94, 1.13], [-0.11, .99]])
W_k = np.array([[0.45, 0.67], [-0.56, 0.78], [1.23, -0.34]])
interactive_qk_vectors(x_1, x_2, W_q, W_k)

Both query vectors ($q_1$ and $q_2$) rotate together when you adjust $W_q$, because the **same weight matrix is shared across all tokens**. $W_q$ defines a fixed weighted combination of embedding features such as "is a verb", "is plural", or "appears early in the sentence" that every token uses to form its query. It cannot use one combination of features for some tokens and a different combination for others.
 
As a consequence, maximising the alignment between $q_1$ and $k_2$ automatically changes the alignment between every other pair in the sequence. A single attention head can therefore only capture one type of inter-token relationship at a time.
 
This is the motivation for **multi-head attention** (Section 5.5): by running $h$ attention heads in parallel, each with its own independent $W_q^{(h)}, W_k^{(h)}, W_v^{(h)}$, the model can specialise each head in a different type of relationship. For example, one head may learn syntactic dependencies, another co-reference, and another positional proximity. In other words, it can simultaneously learn to focus on different aspects of the sequence.

In the demonstration above, $x_1$ and $x_2$ are projected to query–key pairs $(q_1, k_1)$ and $(q_2, k_2)$. Because $W_q$ is shared, maximising alignment between $q_1$ and $k_2$ also changes the alignment between every other pair. A single head imposes one shared geometric constraint across the whole sequence. But note that $q_1 \cdot k_2$ is not the equal to $q_2 \cdot k_1$, i.e. $x_1$ attends differently to $x_2$ than $x_2$ to $x_1$.

Next, we compute the **value** projection. Whereas queries and keys determine *how much* each token attends to another, values determine *what information* is actually passed. The value vector is a linear projection of the input embedding using a learned weight matrix $W_v$:

\begin{equation*}
    v_i = x_i W_v
\end{equation*}

$W_v$ has shape $(d, d_v)$. The value dimension $d_v$ can in principle differ from the head dimension $d_h$, but is typically set equal.

Run the interactive below to see all three projections — query, key, and value — visualised together.


In [ ]:
from src.utils import interactive_qkv_vectors
W_v = np.array([[0.34, -0.56], [0.78, 0.12], [-0.45, 1.23]])
interactive_qkv_vectors(x_1, x_2, W_q, W_k, W_v)

#### 5.4.3 Computing output value vectors through attention 
Attention between $x_i$ and the other vectors $x_j$ in the input context is given by the scores:

\begin{equation*}
    A_i = \big[\, q_i \cdot k_j \,\big]_{j=1}^{N}
\end{equation*}

where $N$ is the size of the input sequence. These raw scores are normalised with a softmax to produce attention weights that sum to one:

\begin{equation*}
    \alpha_{ij} = \mathrm{softmax}_j\!\left(\frac{q_i \cdot k_j}{\sqrt{d_h}}\right)
              = \frac{\exp(q_i \cdot k_j / \sqrt{d_h})}{\sum_{l=1}^{N} \exp(q_i \cdot k_l / \sqrt{d_h})}
\end{equation*}

Finally, the output for token $i$ is the weighted average of all value vectors:

\begin{equation*}
    v_{i,\text{out}} = \sum_{j=1}^{N} \alpha_{ij}\, v_{j}
\end{equation*}

Run the next code cell to understand how context informs output value vectors. 

In [ ]:
from src.utils import interactive_attention

interactive_attention()

#### 5.4.4 Projecting the attention output back to embedding space

The attention output for each token, $v_{i,\text{out}} = \sum_j \alpha_{ij} v_j$, is the
weighted sum of value vectors computed in the previous step. It lives in the value
space ($\mathbb{R}^{d_v}$) and must be projected back to the original embedding space
($\mathbb{R}^{d}$) before it can be passed on. A learned **output projection** $W_O$
does this:

\begin{equation*}
    x_{i,\text{out}} = v_{i,\text{out}} W_O 
\end{equation*}

where $W_O$ has shape $(d_v, d)$, the bias $b$ has shape $(1, d)$. 

Run the next code cell to see the attention output reflected back into the embedding space.

In [ ]:
from src.utils import interactive_attention_with_output

interactive_attention_with_output()

#### 5.4.5 Self-attention head: putting it all together

The four operations we have built up — (1) projecting inputs to query/key/value space, (2) computing scaled dot-product attention scores, (3) taking the weighted average of value vectors, and (4) projecting the output back to embedding space — can be assembled into a single `SelfAttentionHead` PyTorch module. 

Attention is computed from the query-key vectors from all tokens present in a sequence. When using this to model language, there is a problem. In language modelling, the goal of the network is to predict the next token given a sequence of tokens. By allowing the model to attend to all tokens in the sequence, it can "cheat" quite easily. To prevent this we apply a mask for the top-right half of the attention matrix. This is called causal-masking. The following figure illustrates causal masking: 
![causal masking](./images/causal_mask.png)


The `project_output` flag controls whether the internal $W_O$ projection is applied inside the head. When this head is used as a component inside `MultiHeadAttention`, we set `project_output=False` and apply a single shared projection after concatenating all heads.


In [ ]:
import numpy as np 
import torch
import torch.nn as nn

class SelfAttentionHead(nn.Module):
    def __init__(self, d_input, d_model,
                 project_output=True):
        """
        project_output : if True, apply an internal W_O mapping d_model -> d_input
                         (standalone head). Set False when used inside
                         MultiHeadAttention, where a single shared projection is
                         applied after concatenating the heads.
        """
        super().__init__()
        self.d_input = d_input
        self.d_model = d_model
        self.project_output = project_output

        self.W_Q = nn.Linear(d_input, d_model)
        self.W_K = nn.Linear(d_input, d_model)
        self.W_V = nn.Linear(d_input, d_model)
        self.W_O = nn.Linear(d_model, d_input) if project_output else None


    def forward(self, x, forward_mask=None, verbose=False, return_attention_weights=False):
        Q = self.W_Q(x); K = self.W_K(x); V = self.W_V(x)
        if verbose:
            print("Query shape:", Q.shape, " Key:", K.shape, " Value:", V.shape)

        A = Q @ K.transpose(-1, -2) / np.sqrt(self.d_model)
        if forward_mask is not None:
            A = A.masked_fill(~forward_mask, float("-inf"))
        A = A - A.amax(dim=-1, keepdim=True)
        A = torch.exp(A)
        A = A / A.sum(dim=-1, keepdim=True)

        output = A @ V                       # (batch, seq, d_model)
        if self.W_O is not None:
            output = self.W_O(output)        # -> (batch, seq, d_input)

        if return_attention_weights:
            return output, A
        return output


In [ ]:
# Example usage
d_input = 12
d_model = 6
self_attention = SelfAttentionHead(d_input, d_model)
batch_size = 1
seq_length = 4
# Random input embedding vector
x = np.random.rand(batch_size, seq_length, d_input)
x = torch.tensor(x, dtype=torch.float32)
print("x shape:", x.shape)  # Should be (batch_size, seq_length, d_input)
output, attention_weights = self_attention.forward(x, verbose=True, return_attention_weights=True)

print("Output shape:", output.shape)  # Should be (batch_size, seq_length, d_input)
print("Attention weights shape:", attention_weights.shape)  # Should be (batch_size, seq_length, seq_length)



In [ ]:
attention_weights.detach().numpy()

In [ ]:
# show heatmap of attention weights
import seaborn as sns
import matplotlib.pyplot as plt

attention_numpy = attention_weights.detach().numpy()[0]  # (seq_length, seq_length)
sns.heatmap(attention_numpy, annot=True, cmap="turbo")
plt.xlabel("Key positions")
plt.ylabel("Query positions")
plt.title("Attention Weights Heatmap")

### 5.5 Multi-head attention

A single attention head, as we saw in Section 5.1.2, applies one shared $W_q$ across all tokens, allowing it to capture only one type of inter-token relationship at a time. **Multi-head attention** runs $n_{\text{heads}}$ independent attention heads in parallel, each with its own projection matrices $W_q^{(h)}, W_k^{(h)}, W_v^{(h)}$.

Each head can specialise in a different aspect of the sequence: one head might track local sequential patterns while another captures long-range semantic similarity. In proteins, one head might attend to adjacent residues while another attends to residues that are spatially close in the folded 3D structure.

To keep the total computation budget fixed, the model dimension $d_{\text{model}}$ is divided evenly across heads: each head operates in a $d_{\text{head}} = d_{\text{model}} / n_{\text{heads}}$ dimensional space. The outputs of all heads are concatenated along the feature dimension (giving shape $(B, T, d_{\text{model}})$) and projected back to the embedding dimension through a shared $W_O$ matrix.


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, n_heads, d_input, d_model,
                 dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.n_heads = n_heads
        self.d_input = d_input
        self.d_model = d_model
        self.d_head = d_model // n_heads

        # each head projects d_input -> d_head and does NOT apply its own W_O
        self.heads = nn.ModuleList([
            SelfAttentionHead(d_input, self.d_head, project_output=False)
            for _ in range(n_heads)
        ])
        # single shared output projection after concatenation: d_model -> d_input
        self.project_out = nn.Linear(d_model, d_input)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, forward_mask=None, verbose=False, return_attention_weights=False):
        head_outputs = []
        attn_weights = []
        for head in self.heads:
            if return_attention_weights:
                out_h, A_h = head(x, forward_mask, verbose, return_attention_weights)
                attn_weights.append(A_h)
            else:
                out_h = head(x, forward_mask, verbose)
            head_outputs.append(out_h)          # each (batch, seq, d_head)

        # concat heads along feature axis -> (batch, seq, d_model)
        concat = torch.cat(head_outputs, dim=-1)
        output = self.project_out(concat)       # -> (batch, seq, d_input)
        output = self.dropout(output)

        if return_attention_weights:
            attn = torch.stack(attn_weights, dim=1)
            return output, attn
        return output

In [ ]:
msa_example = MultiHeadAttention(n_heads=2, d_input=d_input, d_model=d_model)
x = np.random.rand(batch_size, seq_length, d_input)
x = torch.tensor(x, dtype=torch.float32)
output, attention_weights = msa_example.forward(x, verbose=False, return_attention_weights=True)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")


In [ ]:
# Show two attention heatmaps, one per head

attention_numpy = attention_weights.detach().numpy()[0]  # (n_heads, seq_length, seq_length)
for i in range(attention_numpy.shape[0]):
    plt.figure()
    sns.heatmap(attention_numpy[i], annot=True, cmap="turbo")
    plt.xlabel("Key positions")
    plt.ylabel("Query positions")
    plt.title(f"Attention Weights Heatmap - Head {i+1}")
    

### 5.6 Transformer block

A **transformer block** wraps multi-head attention inside a residual network with two sub-layers:

1. **Multi-head self-attention**: tokens exchange contextual information with every other token.
2. **Position-wise feed-forward network (FFN)**: a two-layer MLP applied independently to each token. The hidden dimension is typically expanded to $4d$ before being compressed back:

$$\text{FFN}(x) = \text{ReLU}(x W_1 + b_1)\, W_2 + b_2, \quad W_1 \in \mathbb{R}^{d \times 4d},\ W_2 \in \mathbb{R}^{4d \times d}$$

Each sub-layer is wrapped with a **residual connection** and **layer normalisation**:

$$x \leftarrow \text{LayerNorm}(x + \text{SubLayer}(x))$$

Residual connections allow gradients to bypass sub-layers during backpropagation, enabling networks with tens or hundreds of blocks to train stably. Layer normalisation stabilises the distribution of activations across tokens and prevents the vanishing and exploding gradients that afflict very deep networks.

Stacking multiple transformer blocks gives the model the capacity to build increasingly abstract representations: early blocks tend to capture local patterns, while later blocks encode long-range and abstract relationships.


In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout()
    
    def forward(self, x):
        x = self.linear1(x)
        x = torch.relu(x)
        x = self.dropout(x)
        x = self.linear2(x)
        return x

class TransformerBlock(nn.Module):
    def __init__(self, n_heads, d_input, d_model, d_ff):
        super().__init__()
        self.attention = MultiHeadAttention(n_heads, d_input, d_model)
        self.norm1 = nn.LayerNorm(d_input)
        self.ffn = FeedForward(d_input, d_ff)
        self.norm2 = nn.LayerNorm(d_input)

    def forward(self, x, forward_mask=None, return_attention_weights=False):
        if return_attention_weights:
            attn_output, attns = self.attention(x, 
                                                forward_mask=forward_mask,
                                                return_attention_weights=return_attention_weights
            )
        else:
            attn_output = self.attention(x, 
                                         forward_mask=forward_mask,
            )
        x = self.norm1(x + attn_output)  # Residual connection + LayerNorm
        ffn_output = self.ffn(x)
        x = self.norm2(x + ffn_output)    # Residual connection + LayerNorm
        if return_attention_weights:
            return x, attns
        else:
            return x

In [ ]:
transformer_block = TransformerBlock(
    n_heads=2,
    d_input=d_input,
    d_model=d_model,
    d_ff=4 * d_input
)
x = np.random.rand(batch_size, seq_length, d_input)
x = torch.tensor(x, dtype=torch.float32)
output, attention_weights = transformer_block(x, return_attention_weights=True)
print(f"Input shape: {x.shape}")
print(f"Transformer block output shape: {output.shape}")


In [ ]:
attention_weights_numpy = attention_weights.detach().numpy()[0]  # (n_heads, seq_length, seq_length)
fig, ax = plt.subplots(1, ncols=2, figsize=(6, 3))
for i in range(attention_weights_numpy.shape[0]):
    sns.heatmap(attention_weights_numpy[i], annot=True, cmap="turbo", ax=ax[i])
    ax[i].set_xlabel("Key positions")
    ax[i].set_ylabel("Query positions")
    ax[i].set_title(f"Head {i+1}")

fig.tight_layout()

### 5.7 Language model

A language model chains multiple transformer blocks in series. Each block refines the token representations: early blocks capture local patterns, while deeper blocks encode progressively more abstract, long-range relationships.

The table below gives a sense of the scale of well-known transformer models:

| Model | Layers | Heads | $d_{\text{model}}$ | Parameters |
|-------|--------|-------|---------------------|------------|
| GPT-2 (small) | 12 | 12 | 768 | 117 M |
| GPT-2 (large) | 36 | 20 | 1 280 | 774 M |
| ESM-2 (150 M) | 30 | 20 | 640 | 150 M |
| ESM-2 (650 M) | 33 | 20 | 1 280 | 650 M |

So far we have been passing a generic embedding tensor $x$ directly into the model. In a real language model, the input is a sequence of **token indices**. Two additional learned components convert these indices into embedding vectors:

- **Token embedding table**: a matrix of shape $(\text{vocab\_size},\ d_{\text{input}})$. Each token index looks up a row in this table to retrieve its embedding vector.
- **Positional embedding table**: a matrix of shape $(\text{context\_length},\ d_{\text{input}})$. Because the attention operation treats its input as a *set* (it is permutation invariant), we must explicitly tell the model where each token sits by adding a learned positional embedding.

Both tables are initialised randomly and updated during training.


In [ ]:
class LanguageModel(nn.Module):
    def __init__(self, n_layers, n_head, d_model, d_input, d_ff, vocab_size, context_length):
        super().__init__()
        self.n_layers = n_layers
        self.n_head = n_head
        self.d_model = d_model 
        self.d_input = d_input 
        self.d_ff = d_ff 

        self.token_embedding = nn.Embedding(vocab_size, d_input) 
        self.positional_embedding = nn.Embedding(context_length, d_input)


        
        self.transformer_layers = nn.ModuleList([TransformerBlock(n_head, d_input, d_model, d_ff) 
                                        for _ in range(n_layers)])
        self.output_head = nn.Linear(d_input, vocab_size)

    def forward(self, idx, forward_mask=None, verbose=False, return_attention_weights=False):
        layer_attentions = []
        token_emb = self.token_embedding(idx)
        positions = torch.arange(idx.shape[1], device=idx.device)
        pos_emb = self.positional_embedding(positions)
        x = token_emb + pos_emb
        layer_out = x
        if return_attention_weights:
            for layer in self.transformer_layers:
                layer_out, layer_attn = layer(layer_out, 
                                              forward_mask=forward_mask,
                                              return_attention_weights=return_attention_weights
                )
                layer_attentions.append(layer_attn)
            
            out = self.output_head(layer_out)
            return out, layer_attentions
        else: 
            for layer in self.transformer_layers:
                layer_out = layer(layer_out,
                                  forward_mask=forward_mask,
                )
            out = self.output_head(layer_out)
            return out
            

In [ ]:
n_layers = 4
n_head = 2
lm_example = LanguageModel(n_layers=n_layers, n_head=n_head, 
                           d_model=d_model, d_input=d_input, 
                           d_ff=4*d_input, vocab_size=50, context_length=10
                        )
idx = torch.tensor([[1, 3, 4, 5]])
lm_output, attns = lm_example(idx, return_attention_weights=True)

print(f"Input shape: {idx.shape}")
print(f"Output shape: {lm_output.shape}")

In [ ]:
attns[0].shape

In [ ]:
fig, ax = plt.subplots(nrows=n_layers, ncols=n_head, figsize=(8, 12))
for layer_num in range(n_layers):
    attentions_this_layer = attns[layer_num]
    for head_num in range(n_head):
        head_attention = attentions_this_layer[0,head_num]
        head_attention_numpy = head_attention.detach().numpy()
        sns.heatmap(head_attention_numpy, annot=True, cmap='turbo', ax=ax[layer_num, head_num])
        title_text = f"Layer: {layer_num+1}, Head: {head_num+1}"
        ax[layer_num, head_num].set_title(title_text)
fig.tight_layout()

In the next module, we will learn how to train language models from a large corpus of data. Since training on the entire database of UniProt sequences is not feasible, we will only train a subset of them. But it is still good to reflect on what structural features are learnt by training a large model. 

### 5.7 Language Models capture long range relationships in sequence data

Does this architecture really capture any meaningful structural information present in the data? For this, we can have a look at an example. We will use the ESM-2 model and see what happens when inputs flow through the network. We will observe the attention maps between layers. 



### 5.7.1 ESM-2: a protein language model trained at scale

The language model we built above is intentionally tiny — it lets us understand the architecture without waiting hours for training. Real protein language models are trained on hundreds of millions of sequences with hundreds of millions of parameters.

**ESM-2** (Evolutionary Scale Modelling 2, Lin et al. 2023) is a family of protein language models developed by Meta AI. The largest variant has 15 billion parameters and was trained on the full UniRef50 database (~250 million unique protein sequences) using masked language modelling. This is the the same objective you will implement in the next notebook.

A striking finding is that ESM-2 attention maps correlate with the physical 3D structure of proteins, even though the model was never shown any structural data. Here we load the 650 M parameter variant and inspect its attention maps for a well-known protein.


In [ ]:
import os 
os.environ["HF_HOME"] = "/projects/nb4170/esm_models"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
import torch
from transformers import AutoModelForMaskedLM 

model_name = "facebook/esm2_t33_650M_UR50D"

model = AutoModelForMaskedLM.from_pretrained(model_name, attn_implementation="eager")
if torch.cuda.is_available():
    device="cuda"
elif torch.backends.mps.is_available():
    device="mps"
else:
    device="cpu"

print(f"Using device: {device}")
model.to(device).eval();


#### What is a contact map?

A **contact map** is a 2-D binary matrix of size $L \times L$ (where $L$ is the sequence length). Entry $(i, j)$ is 1 if residues $i$ and $j$ are within a threshold distance in the folded 3D structure (typically 8 Å between $C_\beta$ atoms), and 0 otherwise. Contact maps encode the spatial proximity relationships of a protein without referencing the full 3D coordinates.

We use **ubiquitin** (PDB: 1UBQ), a small 76-residue protein that is highly conserved across all eukaryotes and tags proteins for degradation. Its compact, well-determined structure makes it an ideal test case. The code below loads the PDB file, computes the contact map from the 3D coordinates, and tokenises the sequence for ESM-2.


In [ ]:
from src.esm_utils import tokenise_pdb_for_esm, pdb_to_contact_map, display_heatmap

pdb_path = "/projects/nb4170/structures/1UBQ.pdb"
tokenised_pdb = tokenise_pdb_for_esm(pdb_path, model_name=model_name)
contact_map = pdb_to_contact_map(pdb_path, distance=8)["contacts"]

display_heatmap(contact_map, tokenised_pdb)


The shared folder contains other PDB files that you can load and inspect: ```1PGB.pdb``` and ```5PTI.pdb```. Try loading them and comparing their contact maps to that of ubiquitin.


#### Extracting attention maps from ESM-2

Passing `output_attentions=True` instructs the model to return the attention weight matrices from every layer. After the forward pass, `out.attentions` is a tuple of length $n_{\text{layers}}$. Each element has shape $(1,\ n_{\text{heads}},\ L,\ L)$: one attention matrix per head per token pair.

We can then compare any individual attention map, selected by layer and head, directly against the contact map. Both are $L \times L$ matrices, so they can be visualised side-by-side. Try changing `layer_to_plot` and `head_to_plot` below to explore which parts of the network show the strongest correspondence with 3D structure.


In [ ]:
input_tensor = tokenised_pdb['input_ids'].to(device)
with torch.no_grad():
    out = model(input_tensor, output_attentions=True)

In [ ]:
attentions = out.attentions

In [ ]:
from esm_utils import display_heatmaps
layer_to_plot = 2  # num_layers = 33, 0-indexed
head_to_plot = 4 # num_heads = 20, 0-indexed


attention_to_plot = attentions[layer_to_plot][0,head_to_plot].detach().numpy()
print(f"Showing attention at Layer: {layer_to_plot}, Head: {head_to_plot}")
print(f"Shape of the attention matrix: {attention_to_plot.shape}")
titles_to_display = [f"Attention - Layer {layer_to_plot+1}, Head {head_to_plot+1}", "Contact Map"]
display_heatmaps([attention_to_plot, contact_map], tokenised_pdb, titles=titles_to_display)


## Assignment 2: Evaluating structural information in ESM-2 attention maps

ESM-2 was trained solely on protein sequences. No 3D structural information was provided during training. Yet the attention maps you have seen above resemble physical contact maps. Your task is to **quantify** this relationship systematically.

### Part 1: Define a similarity measure

Both the attention map and the contact map are 2-D arrays of the same shape. Define a function that returns a scalar score quantifying how similar they are:

```python
def similarity(attention_map, contact_map):
    # your code here
    return score
```

Consider metrics such as **Pearson** or **Spearman correlation**, or **precision-at-L** (the fraction of top-$L$ attention pairs that are true contacts, where $L$ is the sequence length). Briefly justify your choice in a markdown cell.

### Part 2: Distribution of scores across heads

For **at least three layers** (one early, one middle, one late), compute the similarity score for every attention head in that layer and plot a histogram of scores per layer.

> What is the distribution of scores like? Are most heads equally informative, or do a few stand out?

### Part 3: Similarity as a function of layer depth

Aggregate per-head scores using **(a) the mean** and **(b) the maximum** across heads. Plot both curves against layer number on a single graph (layer number on the x-axis, similarity score on the y-axis).

> What does the trend tell you about where structural information is encoded in the network? Does this match your intuition about how depth affects representations in neural networks?
